# Evidencia de Aprendizaje 1: Análisis y herramientas de extracción de datos
## Web scraping con BeautifulSoup, Selenium y Scrapy

**Caso de estudio:** catálogo de libros de práctica *Books to Scrape*  
**Objetivo:** extraer y comparar título, precio, disponibilidad y enlace de productos publicados en un catálogo diseñado para practicar scraping.
#
**Estudiante:** Duván Oswaldo Ramírez Duarte

**Curso:** Programación para análisis de datos

**Docente:** Andrés Felipe Palacio

IU Digital de Antioquia


## 1. Instalación de Google Chrome y dependencias
Esta celda instala Google Chrome estable (evita el lanzador de Chromium Snap que dio error en Colab) y las tres herramientas de extracción.
Ejecuta esta celda primero y espera a que termine.


In [9]:
# Instalar Google Chrome estable para Google Colab
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/google-chrome.deb
!apt-get install -y -qq /tmp/google-chrome.deb

# Instalar bibliotecas del proyecto sin actualizar pandas ni requests
%pip -q install beautifulsoup4 scrapy selenium

# Comprobar que Chrome quedó disponible
!google-chrome --version


Google Chrome 156.0.8078.47 


## 2. Configuración y funciones de apoyo

In [10]:
import sys, subprocess, json, os, re, time
from urllib.parse import urljoin
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL_BASE = "https://books.toscrape.com/"
URL_CATALOGO = urljoin(URL_BASE, "catalogue/page-1.html")
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; StudentScrapingExercise/1.0)"}

def obtener_html(url, timeout=25):
    respuesta = requests.get(url, headers=HEADERS, timeout=timeout)
    respuesta.raise_for_status()
    return respuesta.text

def limpiar_texto(valor):
    return re.sub(r"\s+", " ", valor or "").strip()

print("Configuración lista.")
print("Página seleccionada:", URL_CATALOGO)

Configuración lista.
Página seleccionada: https://books.toscrape.com/catalogue/page-1.html


## 3. Método 1: BeautifulSoup
Se descarga el HTML con `requests` y se analizan sus elementos con BeautifulSoup.

In [11]:
html = obtener_html(URL_CATALOGO)
soup = BeautifulSoup(html, "html.parser")

resultados_bs = []
for articulo in soup.select("article.product_pod"):
    enlace = articulo.select_one("h3 a")
    precio = articulo.select_one(".price_color")
    disponibilidad = articulo.select_one(".availability")
    if enlace:
        resultados_bs.append({
            "titulo": limpiar_texto(enlace.get("title") or enlace.get_text()),
            "precio": limpiar_texto(precio.get_text()) if precio else None,
            "disponibilidad": limpiar_texto(disponibilidad.get_text()) if disponibilidad else None,
            "url": urljoin(URL_CATALOGO, enlace.get("href", ""))
        })

df_bs = pd.DataFrame(resultados_bs)
print("Productos extraídos con BeautifulSoup:", len(df_bs))
display(df_bs.head(10))
assert len(df_bs) > 0, "No se encontraron productos; verifica la conexión o la estructura de la página."

Productos extraídos con BeautifulSoup: 20


,titulo,precio,disponibilidad,url
0,A Light in the Attic,Â£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,Â£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,Â£52.15,In stock,https://books.toscrape.com/catalogue/the-black...


## 4. Método 2: Selenium
Selenium controla Google Chrome en modo headless. En este notebook se utiliza Google Chrome estable y Selenium Manager para resolver el controlador compatible.


In [12]:
# Selenium probado para Google Colab: Google Chrome + Selenium Manager
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

resultados_sel = []
driver = None

opciones = Options()
opciones.binary_location = "/usr/bin/google-chrome"
opciones.add_argument("--headless=new")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")
opciones.add_argument("--disable-gpu")
opciones.add_argument("--window-size=1920,1080")

try:
    driver = webdriver.Chrome(options=opciones)
    driver.set_page_load_timeout(60)
    driver.get(URL_CATALOGO)

    WebDriverWait(driver, 30).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "article.product_pod")
        )
    )

    articulos = driver.find_elements(
        By.CSS_SELECTOR, "article.product_pod"
    )

    for articulo in articulos:
        enlace = articulo.find_element(By.CSS_SELECTOR, "h3 a")
        precio = articulo.find_element(By.CSS_SELECTOR, ".price_color")
        disponibilidad = articulo.find_element(By.CSS_SELECTOR, ".availability")
        resultados_sel.append({
            "titulo": limpiar_texto(enlace.get_attribute("title") or enlace.text),
            "precio": limpiar_texto(precio.text),
            "disponibilidad": limpiar_texto(disponibilidad.text),
            "url": enlace.get_attribute("href")
        })

    print("Título:", driver.title)
    print("Libros encontrados con Selenium:", len(resultados_sel))
    for libro in resultados_sel[:3]:
        print("-", libro["titulo"])

except Exception as error:
    print("ERROR DE SELENIUM:", type(error).__name__)
    print(str(error)[:2500])
    print("No se inventan resultados: df_selenium quedará vacío si la extracción falla.")

finally:
    if driver is not None:
        driver.quit()

df_selenium = pd.DataFrame(
    resultados_sel,
    columns=["titulo", "precio", "disponibilidad", "url"]
)
print("Filas finales de Selenium:", len(df_selenium))
display(df_selenium.head(10))


Título: All products | Books to Scrape - Sandbox
Libros encontrados con Selenium: 20
- A Light in the Attic
- Tipping the Velvet
- Soumission
Filas finales de Selenium: 20


,titulo,precio,disponibilidad,url
0,A Light in the Attic,£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,£52.15,In stock,https://books.toscrape.com/catalogue/the-black...


## 5. Método 3: Scrapy
Se crea una araña Scrapy que analiza el HTML con selectores CSS y visita una página de catálogo. Se ejecuta como un proceso separado para reducir conflictos con el reactor de Colab.

In [13]:
import tempfile, pathlib, subprocess, sys, os, json

spider_code = r'''
import scrapy
import json
from urllib.parse import urljoin

class CatalogoSpider(scrapy.Spider):
    name = "catalogo_practica"
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]
    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.5,
        "USER_AGENT": "StudentScrapingExercise/1.0",
        "LOG_LEVEL": "ERROR",
        "FEEDS": {}
    }

    def parse(self, response):
        for articulo in response.css("article.product_pod"):
            enlace = articulo.css("h3 a")
            titulo = enlace.attrib.get("title") or " ".join(enlace.css("::text").getall()).strip()
            yield {
                "titulo": titulo,
                "precio": " ".join(articulo.css(".price_color::text").getall()).strip(),
                "disponibilidad": " ".join(articulo.css(".availability::text").getall()).strip(),
                "url": response.urljoin(enlace.attrib.get("href", ""))
            }
'''

with tempfile.TemporaryDirectory() as carpeta:
    ruta_spider = os.path.join(carpeta, "catalogo_spider.py")
    ruta_json = os.path.join(carpeta, "resultados.json")
    with open(ruta_spider, "w", encoding="utf-8") as archivo:
        archivo.write(spider_code)

    comando = [
        sys.executable, "-m", "scrapy", "runspider", ruta_spider,
        "-O", ruta_json, "-s", "LOG_LEVEL=ERROR"
    ]
    proceso = subprocess.run(comando, capture_output=True, text=True, timeout=120)
    if proceso.returncode == 0 and os.path.exists(ruta_json):
        with open(ruta_json, "r", encoding="utf-8") as archivo:
            resultados_scrapy = json.load(archivo)
    else:
        resultados_scrapy = []
        print("Scrapy no pudo completar la extracción.")
        print((proceso.stderr or proceso.stdout)[-1500:])

df_scrapy = pd.DataFrame(resultados_scrapy, columns=["titulo", "precio", "disponibilidad", "url"])
print("Productos extraídos con Scrapy:", len(df_scrapy))
if not df_scrapy.empty:
    display(df_scrapy.head(10))

Productos extraídos con Scrapy: 20


,titulo,precio,disponibilidad,url
0,A Light in the Attic,£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,£52.15,In stock,https://books.toscrape.com/catalogue/the-black...


## 6. Validación y comparación
Se comprueba que cada método haya devuelto registros, se comparan los conteos y se revisan valores vacíos y duplicados.

In [14]:
def resumen_calidad(nombre, df):
    if df.empty:
        return {
            "metodo": nombre, "registros": 0, "titulos_vacios": None,
            "precios_vacios": None, "urls_duplicadas": None,
            "estado": "Sin resultados; revisar el mensaje de la celda"
        }
    return {
        "metodo": nombre,
        "registros": len(df),
        "titulos_vacios": int(df["titulo"].isna().sum() + (df["titulo"].astype(str).str.strip() == "").sum()),
        "precios_vacios": int(df["precio"].isna().sum() + (df["precio"].astype(str).str.strip() == "").sum()),
        "urls_duplicadas": int(df["url"].duplicated().sum()),
        "estado": "OK" if len(df) > 0 else "Sin resultados"
    }

comparacion = pd.DataFrame([
    resumen_calidad("BeautifulSoup", df_bs),
    resumen_calidad("Selenium", df_selenium),
    resumen_calidad("Scrapy", df_scrapy)
])

print("COMPARACIÓN DE LOS TRES MÉTODOS")
display(comparacion)

# Guardar cada extracción y la tabla comparativa como CSV
df_bs.to_csv("resultados_beautifulsoup.csv", index=False, encoding="utf-8-sig")
df_selenium.to_csv("resultados_selenium.csv", index=False, encoding="utf-8-sig")
df_scrapy.to_csv("resultados_scrapy.csv", index=False, encoding="utf-8-sig")
comparacion.to_csv("comparacion_metodos.csv", index=False, encoding="utf-8-sig")

# Empaquetar los cuatro CSV en un ZIP para descargarlos fácilmente desde Colab
import zipfile
with zipfile.ZipFile("resultados_scraping.zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for nombre_archivo in [
        "resultados_beautifulsoup.csv",
        "resultados_selenium.csv",
        "resultados_scrapy.csv",
        "comparacion_metodos.csv"
    ]:
        zipf.write(nombre_archivo)

print("Se guardaron 4 CSV y el archivo resultados_scraping.zip.")


COMPARACIÓN DE LOS TRES MÉTODOS


,metodo,registros,titulos_vacios,precios_vacios,urls_duplicadas,estado
0,BeautifulSoup,20,0,0,0,OK
1,Selenium,20,0,0,0,OK
2,Scrapy,20,0,0,0,OK


Se guardaron 4 CSV y el archivo resultados_scraping.zip.


## 7. Resultados y conclusiones automáticas

In [15]:
print("RESULTADOS")
print(f"BeautifulSoup: {len(df_bs)} productos")
print(f"Selenium: {len(df_selenium)} productos")
print(f"Scrapy: {len(df_scrapy)} productos")

print("\nCONCLUSIONES")
print("- BeautifulSoup es sencillo y adecuado para analizar HTML ya descargado.")
print("- Selenium permite interactuar con un navegador y resulta útil cuando el contenido depende de JavaScript.")
print("- Scrapy facilita organizar rastreos y procesar múltiples páginas de forma estructurada.")
print("- Los conteos pueden variar si hay fallos de red o diferencias del entorno; por eso se incluyen validaciones.")
print("- La extracción debe respetar los términos del sitio, el archivo robots.txt, la privacidad y la carga razonable del servidor.")

RESULTADOS
BeautifulSoup: 20 productos
Selenium: 20 productos
Scrapy: 20 productos

CONCLUSIONES
- BeautifulSoup es sencillo y adecuado para analizar HTML ya descargado.
- Selenium permite interactuar con un navegador y resulta útil cuando el contenido depende de JavaScript.
- Scrapy facilita organizar rastreos y procesar múltiples páginas de forma estructurada.
- Los conteos pueden variar si hay fallos de red o diferencias del entorno; por eso se incluyen validaciones.
- La extracción debe respetar los términos del sitio, el archivo robots.txt, la privacidad y la carga razonable del servidor.


## 9. Descargar los resultados
Ejecuta esta celda al final si quieres descargar un ZIP con los CSV de BeautifulSoup, Selenium, Scrapy y la comparación.

In [16]:
from google.colab import files
files.download('resultados_scraping.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Bibliografía (formato APA)

Beautiful Soup. (s. f.). *Beautiful Soup documentation*. https://www.crummy.com/software/BeautifulSoup/bs4/doc/

Scrapy. (s. f.). *Scrapy documentation*. https://docs.scrapy.org/en/latest/

Selenium. (s. f.). *Selenium documentation*. https://www.selenium.dev/documentation/

Books to Scrape. (s. f.). *Books to Scrape: A fictional bookstore for web scraping practice*. https://books.toscrape.com/

## Nota de ejecución
En Google Colab selecciona **Entorno de ejecución → Ejecutar todas**. La celda de Selenium está protegida con manejo de excepciones para que una falla del navegador no detenga las demás secciones. Si una web bloquea las solicitudes, no intentes evadir sus medidas de seguridad: utiliza un sitio de práctica o una API autorizada.
